<a href="https://colab.research.google.com/github/riyapai05/TULU_PROJECT/blob/main/Tulu_NonTulu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CONVNEXT-TINY**

In [ ]:
# ============================================================
# COMPLETE CONVNEXT CODE
# ============================================================

# ============================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install timm -q

# ============================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================

import os
import zipfile
import shutil
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from PIL import Image
import timm
from google.colab import files

# ============================================================
# STEP 3: UPLOAD ZIP FILES
# ============================================================

print("Upload:")
print("1. Tulu_Dataset_1.zip")
print("2. Non Tulu.zip")

uploaded = files.upload()

# ============================================================
# STEP 4: EXTRACT ZIP FILES
# ============================================================

extract_path = "/content/datasets"

os.makedirs(extract_path, exist_ok=True)

for zip_file in uploaded.keys():

    print(f"Extracting {zip_file}...")

    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("\nExtraction Completed!")

# ============================================================
# STEP 5: CHECK EXTRACTED FOLDERS
# ============================================================

print("\nFolders inside dataset:\n")

for item in os.listdir(extract_path):
    print(item)

# ============================================================
# STEP 6: CREATE COMBINED DATASET
# ============================================================

import os
import shutil

combined_path = "/content/combined_dataset"

if os.path.exists(combined_path):
    shutil.rmtree(combined_path)

os.makedirs(combined_path)

# ============================================================
# FIND TULU ROOT AUTOMATICALLY
# ============================================================

tulu_root = os.path.join(
    extract_path,
    "Tulu_Dataset_1"
)

# Handle nested folder after extraction
items = os.listdir(tulu_root)

if len(items) == 1 and os.path.isdir(
    os.path.join(tulu_root, items[0])
):
    possible_nested = os.path.join(
        tulu_root,
        items[0]
    )

    if len(os.listdir(possible_nested)) > 0:
        tulu_root = possible_nested

print("\nTulu Root:", tulu_root)

# ============================================================
# COPY TULU CHARACTER CLASSES
# ============================================================

print("\nProcessing Tulu Dataset...")

total_tulu_images = 0

for character in os.listdir(tulu_root):

    char_path = os.path.join(
        tulu_root,
        character
    )

    if not os.path.isdir(char_path):
        continue

    destination_class = os.path.join(
        combined_path,
        character
    )

    os.makedirs(
        destination_class,
        exist_ok=True
    )

    count = 0

    for image_name in os.listdir(char_path):

        src = os.path.join(
            char_path,
            image_name
        )

        if not os.path.isfile(src):
            continue

        dst = os.path.join(
            destination_class,
            f"tulu_{image_name}"
        )

        shutil.copy(src, dst)

        count += 1
        total_tulu_images += 1

    print(
        f"{character} -> {count} images"
    )

# ============================================================
# NON TULU CLASS
# ============================================================

non_tulu_root = os.path.join(
    extract_path,
    "Non_Tulu"
)

print("\nProcessing Non_Tulu Dataset...")

non_tulu_class = os.path.join(
    combined_path,
    "NON_TULU"
)

os.makedirs(
    non_tulu_class,
    exist_ok=True
)

non_tulu_count = 0

for image_name in os.listdir(non_tulu_root):

    src = os.path.join(
        non_tulu_root,
        image_name
    )

    if not os.path.isfile(src):
        continue

    dst = os.path.join(
        non_tulu_class,
        image_name
    )

    shutil.copy(src, dst)

    non_tulu_count += 1

# ============================================================
# SUMMARY
# ============================================================

print("\n===================================")
print("DATASET CREATED SUCCESSFULLY")
print("===================================")

print(
    f"\nTotal Tulu Images : {total_tulu_images}"
)

print(
    f"Total Non-Tulu Images : {non_tulu_count}"
)

print("\nClasses:\n")

for cls in sorted(os.listdir(combined_path)):

    cls_path = os.path.join(
        combined_path,
        cls
    )

    print(
        f"{cls} -> {len(os.listdir(cls_path))} images"
    )

print(
    "\nTotal Classes:",
    len(os.listdir(combined_path))
)

# ============================================================
# STEP 7: IMAGE TRANSFORMS
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# STEP 8: LOAD DATASET
# ============================================================

dataset = ImageFolder(
    root=combined_path,
    transform=transform
)

print("\nDataset Loaded Successfully!")

print("\nClasses:\n")
print(dataset.classes)

num_classes = len(dataset.classes)

print("\nNumber of Classes:", num_classes)

# ============================================================
# STEP 9: SPLIT DATASET
# ============================================================

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("\nTraining Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))

# ============================================================
# STEP 10: DEVICE SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nUsing Device:", device)

# ============================================================
# STEP 11: LOAD CONVNEXT MODEL
# ============================================================

model = timm.create_model(
    "convnext_tiny",
    pretrained=True,
    num_classes=num_classes
)

model.to(device)

print("\nConvNeXt Model Loaded!")

# ============================================================
# STEP 12: LOSS FUNCTION & OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001
)

# ============================================================
# STEP 13: TRAIN MODEL WITH LIVE PROGRESS
# ============================================================

epochs = 10

for epoch in range(epochs):

    print(f"\n================ EPOCH {epoch+1}/{epochs} ================\n")

    # ================= TRAIN =================

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

        # LIVE BATCH PRINT
        if (batch_idx + 1) % 10 == 0:

            batch_accuracy = 100 * correct / total

            print(
                f"Batch [{batch_idx+1}/{len(train_loader)}] "
                f"| Loss: {loss.item():.4f} "
                f"| Accuracy: {batch_accuracy:.2f}%"
            )

    train_accuracy = 100 * correct / total

    # ================= VALIDATION =================

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_accuracy = 100 * val_correct / val_total

    print("\n========================================")

    print(f"Epoch [{epoch+1}/{epochs}] Completed")

    print(f"Train Loss       : {running_loss:.4f}")

    print(f"Train Accuracy   : {train_accuracy:.2f}%")

    print(f"Validation Acc   : {val_accuracy:.2f}%")

    print("========================================")

# ============================================================
# STEP 14: SAVE MODEL
# ============================================================

torch.save(model.state_dict(), "tulu_nontulu_convnext.pth")

print("\nModel Saved Successfully!")

# ============================================================
# STEP 15: SAVE CLASS NAMES
# ============================================================

class_names = dataset.classes

with open("class_names.txt", "w") as f:

    for item in class_names:
        f.write(item + "\n")

print("Class Names Saved!")

Upload:
1. Tulu_Dataset_1.zip
2. Non Tulu.zip


Saving Non_Tulu.zip to Non_Tulu.zip
Saving Tulu_Dataset_1.zip to Tulu_Dataset_1.zip
Extracting Non_Tulu.zip...
Extracting Tulu_Dataset_1.zip...

Extraction Completed!

Folders inside dataset:

Tulu_Dataset_1
Non_Tulu

Tulu Root: /content/datasets/Tulu_Dataset_1

Processing Tulu Dataset...
ಜ -> 70 images
ಔ -> 73 images
ಮ -> 73 images
ಪ -> 72 images
ಕ -> 73 images
ಢ -> 67 images
ವ -> 74 images
ಙ -> 73 images
ಠ -> 77 images
ಧ -> 72 images
ಡ -> 73 images
ಥ -> 72 images
ಒ -> 73 images
ಸ -> 73 images
ಲ -> 73 images
ಳ -> 74 images
ಐ -> 73 images
ತ -> 72 images
ಝ -> 73 images
ಛ -> 73 images
ಷ -> 73 images
ಆ -> 74 images
ಗ -> 73 images
ಋ -> 73 images
ರ -> 73 images
ಅಃ -> 73 images
ಈ -> 73 images
ಊ -> 73 images
ಟ -> 75 images
ಓ -> 73 images
ೠ -> 67 images
ಬ -> 73 images
ಶ -> 64 images
ಞ -> 74 images
ಯ -> 70 images
ನ -> 72 images
ಅಂ -> 73 images
ಣ -> 73 images
ಎ -> 73 images
ಹ -> 73 images
ಇ -> 72 images
ಫ -> 72 images
ಏ -> 73 images
ಘ -> 70 images
ಖ -> 73 images
ಚ -> 73 images
ಉ -> 73 images
ದ -

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]


ConvNeXt Model Loaded!

================ EPOCH 1/10 ================

Batch [10/170] | Loss: 3.0278 | Accuracy: 35.94%
Batch [20/170] | Loss: 2.2504 | Accuracy: 41.09%
Batch [30/170] | Loss: 2.8241 | Accuracy: 42.71%
Batch [40/170] | Loss: 2.0906 | Accuracy: 44.92%
Batch [50/170] | Loss: 1.9909 | Accuracy: 45.12%
Batch [60/170] | Loss: 2.7427 | Accuracy: 44.90%
Batch [70/170] | Loss: 2.6058 | Accuracy: 45.00%
Batch [80/170] | Loss: 2.6187 | Accuracy: 45.16%
Batch [90/170] | Loss: 2.1748 | Accuracy: 45.73%
Batch [100/170] | Loss: 1.7497 | Accuracy: 46.62%
Batch [110/170] | Loss: 2.0089 | Accuracy: 47.33%
Batch [120/170] | Loss: 0.9389 | Accuracy: 48.36%
Batch [130/170] | Loss: 0.6005 | Accuracy: 50.07%
Batch [140/170] | Loss: 0.5972 | Accuracy: 52.03%
Batch [150/170] | Loss: 0.7215 | Accuracy: 53.44%
Batch [160/170] | Loss: 0.5154 | Accuracy: 55.10%
Batch [170/170] | Loss: 0.3779 | Accuracy: 56.57%

Epoch [1/10] Completed
Train Loss       : 317.6447
Train Accuracy   : 56.57%
Validation

In [ ]:
# ============================================================
# STEP 21: DOWNLOAD MODEL FILES
# ============================================================

files.download("tulu_nontulu_convnext.pth")

files.download("class_names.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
from PIL import Image
import torch

# Upload image
uploaded = files.upload()

image_path = list(uploaded.keys())[0]

# Load image
image = Image.open(image_path).convert("RGB")

# Transform
image = transform(image)

# Add batch dimension
image = image.unsqueeze(0).to(device)

# Prediction
model.eval()

with torch.no_grad():

    outputs = model(image)

    probabilities = torch.softmax(outputs, dim=1)

    confidence, predicted = torch.max(probabilities, 1)

predicted_class = class_names[predicted.item()]
confidence_score = confidence.item() * 100

print("\n==============================")
print("RESULT")
print("==============================")

if predicted_class == "NON_TULU":

    print("Language   : NON_TULU")
    print(f"Confidence : {confidence_score:.2f}%")

else:

    print("Language   : TULU")
    print("Character  :", predicted_class)
    print(f"Confidence : {confidence_score:.2f}%")

Saving pre_62b119eb-c0f5-4733-a595-593ff603ea84.jpg to pre_62b119eb-c0f5-4733-a595-593ff603ea84.jpg

RESULT
Language   : NON_TULU
Confidence : 99.97%


**SWIN**

In [ ]:
# ============================================================
# COMPLETE CONVNEXT CODE
# ============================================================

# ============================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install timm -q

# ============================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================

import os
import zipfile
import shutil
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from PIL import Image
import timm
from google.colab import files

# ============================================================
# STEP 3: UPLOAD ZIP FILES
# ============================================================

print("Upload:")
print("1. Tulu_Dataset_1.zip")
print("2. Non Tulu.zip")

uploaded = files.upload()

# ============================================================
# STEP 4: EXTRACT ZIP FILES
# ============================================================

extract_path = "/content/datasets"

os.makedirs(extract_path, exist_ok=True)

for zip_file in uploaded.keys():

    print(f"Extracting {zip_file}...")

    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("\nExtraction Completed!")

# ============================================================
# STEP 5: CHECK EXTRACTED FOLDERS
# ============================================================

print("\nFolders inside dataset:\n")

for item in os.listdir(extract_path):
    print(item)

# ============================================================
# STEP 6: CREATE COMBINED DATASET
# ============================================================

import os
import shutil

combined_path = "/content/combined_dataset"

if os.path.exists(combined_path):
    shutil.rmtree(combined_path)

os.makedirs(combined_path)

# ============================================================
# FIND TULU ROOT AUTOMATICALLY
# ============================================================

tulu_root = os.path.join(
    extract_path,
    "Tulu_Dataset_1"
)

# Handle nested folder after extraction
items = os.listdir(tulu_root)

if len(items) == 1 and os.path.isdir(
    os.path.join(tulu_root, items[0])
):
    possible_nested = os.path.join(
        tulu_root,
        items[0]
    )

    if len(os.listdir(possible_nested)) > 0:
        tulu_root = possible_nested

print("\nTulu Root:", tulu_root)

# ============================================================
# COPY TULU CHARACTER CLASSES
# ============================================================

print("\nProcessing Tulu Dataset...")

total_tulu_images = 0

for character in os.listdir(tulu_root):

    char_path = os.path.join(
        tulu_root,
        character
    )

    if not os.path.isdir(char_path):
        continue

    destination_class = os.path.join(
        combined_path,
        character
    )

    os.makedirs(
        destination_class,
        exist_ok=True
    )

    count = 0

    for image_name in os.listdir(char_path):

        src = os.path.join(
            char_path,
            image_name
        )

        if not os.path.isfile(src):
            continue

        dst = os.path.join(
            destination_class,
            f"tulu_{image_name}"
        )

        shutil.copy(src, dst)

        count += 1
        total_tulu_images += 1

    print(
        f"{character} -> {count} images"
    )

# ============================================================
# NON TULU CLASS
# ============================================================

non_tulu_root = os.path.join(
    extract_path,
    "Non_Tulu"
)

print("\nProcessing Non_Tulu Dataset...")

non_tulu_class = os.path.join(
    combined_path,
    "NON_TULU"
)

os.makedirs(
    non_tulu_class,
    exist_ok=True
)

non_tulu_count = 0

for image_name in os.listdir(non_tulu_root):

    src = os.path.join(
        non_tulu_root,
        image_name
    )

    if not os.path.isfile(src):
        continue

    dst = os.path.join(
        non_tulu_class,
        image_name
    )

    shutil.copy(src, dst)

    non_tulu_count += 1

# ============================================================
# SUMMARY
# ============================================================

print("\n===================================")
print("DATASET CREATED SUCCESSFULLY")
print("===================================")

print(
    f"\nTotal Tulu Images : {total_tulu_images}"
)

print(
    f"Total Non-Tulu Images : {non_tulu_count}"
)

print("\nClasses:\n")

for cls in sorted(os.listdir(combined_path)):

    cls_path = os.path.join(
        combined_path,
        cls
    )

    print(
        f"{cls} -> {len(os.listdir(cls_path))} images"
    )

print(
    "\nTotal Classes:",
    len(os.listdir(combined_path))
)

# ============================================================
# STEP 7: IMAGE TRANSFORMS
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# STEP 8: LOAD DATASET
# ============================================================

dataset = ImageFolder(
    root=combined_path,
    transform=transform
)

print("\nDataset Loaded Successfully!")

print("\nClasses:\n")
print(dataset.classes)

num_classes = len(dataset.classes)

print("\nNumber of Classes:", num_classes)

# ============================================================
# STEP 9: SPLIT DATASET
# ============================================================

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("\nTraining Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))

# ============================================================
# STEP 10: DEVICE SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nUsing Device:", device)

# ============================================================
# STEP 11: LOAD CONVNEXT MODEL
# ============================================================

model = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=True,
    num_classes=num_classes
)

model.to(device)

print("\nSwin Model Loaded!")

# ============================================================
# STEP 12: LOSS FUNCTION & OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001
)

# ============================================================
# STEP 13: TRAIN MODEL WITH LIVE PROGRESS
# ============================================================

epochs = 10

for epoch in range(epochs):

    print(f"\n================ EPOCH {epoch+1}/{epochs} ================\n")

    # ================= TRAIN =================

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

        # LIVE BATCH PRINT
        if (batch_idx + 1) % 10 == 0:

            batch_accuracy = 100 * correct / total

            print(
                f"Batch [{batch_idx+1}/{len(train_loader)}] "
                f"| Loss: {loss.item():.4f} "
                f"| Accuracy: {batch_accuracy:.2f}%"
            )

    train_accuracy = 100 * correct / total

    # ================= VALIDATION =================

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_accuracy = 100 * val_correct / val_total

    print("\n========================================")

    print(f"Epoch [{epoch+1}/{epochs}] Completed")

    print(f"Train Loss       : {running_loss:.4f}")

    print(f"Train Accuracy   : {train_accuracy:.2f}%")

    print(f"Validation Acc   : {val_accuracy:.2f}%")

    print("========================================")

# ============================================================
# STEP 14: SAVE MODEL
# ============================================================

torch.save(model.state_dict(), "tulu_nontulu_swin.pth")

print("\nModel Saved Successfully!")

# ============================================================
# STEP 15: SAVE CLASS NAMES
# ============================================================

class_names = dataset.classes

with open("class_names.txt", "w") as f:

    for item in class_names:
        f.write(item + "\n")

print("Class Names Saved!")

Upload:
1. Tulu_Dataset_1.zip
2. Non Tulu.zip


Saving Non_Tulu.zip to Non_Tulu.zip
Saving Tulu_Dataset_1.zip to Tulu_Dataset_1.zip
Extracting Non_Tulu.zip...
Extracting Tulu_Dataset_1.zip...

Extraction Completed!

Folders inside dataset:

Tulu_Dataset_1
Non_Tulu

Tulu Root: /content/datasets/Tulu_Dataset_1/Tulu_Dataset_1

Processing Tulu Dataset...
ಜ -> 70 images
ಔ -> 73 images
ಮ -> 73 images
ಪ -> 72 images
ಕ -> 73 images
ಢ -> 67 images
ವ -> 74 images
ಙ -> 73 images
ಠ -> 77 images
ಧ -> 72 images
ಡ -> 73 images
ಥ -> 72 images
ಒ -> 73 images
ಸ -> 73 images
ಲ -> 73 images
ಳ -> 74 images
ಐ -> 73 images
ತ -> 72 images
ಝ -> 73 images
ಛ -> 73 images
ಷ -> 73 images
ಆ -> 74 images
ಗ -> 73 images
ಋ -> 73 images
ರ -> 73 images
ಅಃ -> 73 images
ಈ -> 73 images
ಊ -> 73 images
ಟ -> 75 images
ಓ -> 73 images
ೠ -> 67 images
ಬ -> 73 images
ಶ -> 64 images
ಞ -> 74 images
ಯ -> 70 images
ನ -> 72 images
ಅಂ -> 73 images
ಣ -> 73 images
ಎ -> 73 images
ಹ -> 73 images
ಇ -> 72 images
ಫ -> 72 images
ಏ -> 73 images
ಘ -> 70 images
ಖ -> 73 images
ಚ -> 73 images
ಉ -

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]


Swin Model Loaded!

================ EPOCH 1/10 ================

Batch [10/170] | Loss: 2.4397 | Accuracy: 41.56%
Batch [20/170] | Loss: 2.6698 | Accuracy: 42.97%
Batch [30/170] | Loss: 2.0914 | Accuracy: 44.38%
Batch [40/170] | Loss: 2.5516 | Accuracy: 44.30%
Batch [50/170] | Loss: 2.2777 | Accuracy: 45.06%
Batch [60/170] | Loss: 2.1509 | Accuracy: 45.05%
Batch [70/170] | Loss: 1.9942 | Accuracy: 45.54%
Batch [80/170] | Loss: 1.6094 | Accuracy: 45.86%
Batch [90/170] | Loss: 2.1005 | Accuracy: 46.04%
Batch [100/170] | Loss: 2.5585 | Accuracy: 45.84%
Batch [110/170] | Loss: 1.6698 | Accuracy: 46.51%
Batch [120/170] | Loss: 1.7063 | Accuracy: 47.89%
Batch [130/170] | Loss: 1.6302 | Accuracy: 48.92%
Batch [140/170] | Loss: 1.6643 | Accuracy: 49.82%
Batch [150/170] | Loss: 0.9132 | Accuracy: 51.00%
Batch [160/170] | Loss: 0.7767 | Accuracy: 52.07%
Batch [170/170] | Loss: 1.3785 | Accuracy: 52.84%

Epoch [1/10] Completed
Train Loss       : 338.9091
Train Accuracy   : 52.84%
Validation Acc

In [ ]:
# ============================================================
# STEP 21: DOWNLOAD MODEL FILES
# ============================================================

files.download("tulu_nontulu_swin.pth")

files.download("class_names.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
from PIL import Image
import torch

# Upload image
uploaded = files.upload()

image_path = list(uploaded.keys())[0]

# Load image
image = Image.open(image_path).convert("RGB")

# Transform
image = transform(image)

# Add batch dimension
image = image.unsqueeze(0).to(device)

# Prediction
model.eval()

with torch.no_grad():

    outputs = model(image)

    probabilities = torch.softmax(outputs, dim=1)

    confidence, predicted = torch.max(probabilities, 1)

predicted_class = class_names[predicted.item()]
confidence_score = confidence.item() * 100

print("\n==============================")
print("RESULT")
print("==============================")

if predicted_class == "NON_TULU":

    print("Language   : NON_TULU")
    print(f"Confidence : {confidence_score:.2f}%")

else:

    print("Language   : TULU")
    print("Character  :", predicted_class)
    print(f"Confidence : {confidence_score:.2f}%")

Saving Screenshot 2026-06-02 102424.png to Screenshot 2026-06-02 102424.png

RESULT
Language   : TULU
Character  : ರ
Confidence : 99.91%


**ViT**

In [ ]:
# ============================================================
# COMPLETE vit CODE
# ============================================================

# ============================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install timm -q

# ============================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================

import os
import zipfile
import shutil
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from PIL import Image
import timm
from google.colab import files

# ============================================================
# STEP 3: UPLOAD ZIP FILES
# ============================================================

print("Upload:")
print("1. Tulu_Dataset_1.zip")
print("2. Non Tulu.zip")

uploaded = files.upload()

# ============================================================
# STEP 4: EXTRACT ZIP FILES
# ============================================================

extract_path = "/content/datasets"

os.makedirs(extract_path, exist_ok=True)

for zip_file in uploaded.keys():

    print(f"Extracting {zip_file}...")

    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("\nExtraction Completed!")

# ============================================================
# STEP 5: CHECK EXTRACTED FOLDERS
# ============================================================

print("\nFolders inside dataset:\n")

for item in os.listdir(extract_path):
    print(item)

# ============================================================
# STEP 6: CREATE COMBINED DATASET
# ============================================================

import os
import shutil

combined_path = "/content/combined_dataset"

if os.path.exists(combined_path):
    shutil.rmtree(combined_path)

os.makedirs(combined_path)

# ============================================================
# FIND TULU ROOT AUTOMATICALLY
# ============================================================

tulu_root = os.path.join(
    extract_path,
    "Tulu_Dataset_1"
)

# Handle nested folder after extraction
items = os.listdir(tulu_root)

if len(items) == 1 and os.path.isdir(
    os.path.join(tulu_root, items[0])
):
    possible_nested = os.path.join(
        tulu_root,
        items[0]
    )

    if len(os.listdir(possible_nested)) > 0:
        tulu_root = possible_nested

print("\nTulu Root:", tulu_root)

# ============================================================
# COPY TULU CHARACTER CLASSES
# ============================================================

print("\nProcessing Tulu Dataset...")

total_tulu_images = 0

for character in os.listdir(tulu_root):

    char_path = os.path.join(
        tulu_root,
        character
    )

    if not os.path.isdir(char_path):
        continue

    destination_class = os.path.join(
        combined_path,
        character
    )

    os.makedirs(
        destination_class,
        exist_ok=True
    )

    count = 0

    for image_name in os.listdir(char_path):

        src = os.path.join(
            char_path,
            image_name
        )

        if not os.path.isfile(src):
            continue

        dst = os.path.join(
            destination_class,
            f"tulu_{image_name}"
        )

        shutil.copy(src, dst)

        count += 1
        total_tulu_images += 1

    print(
        f"{character} -> {count} images"
    )

# ============================================================
# NON TULU CLASS
# ============================================================

non_tulu_root = os.path.join(
    extract_path,
    "Non_Tulu"
)

print("\nProcessing Non_Tulu Dataset...")

non_tulu_class = os.path.join(
    combined_path,
    "NON_TULU"
)

os.makedirs(
    non_tulu_class,
    exist_ok=True
)

non_tulu_count = 0

for image_name in os.listdir(non_tulu_root):

    src = os.path.join(
        non_tulu_root,
        image_name
    )

    if not os.path.isfile(src):
        continue

    dst = os.path.join(
        non_tulu_class,
        image_name
    )

    shutil.copy(src, dst)

    non_tulu_count += 1

# ============================================================
# SUMMARY
# ============================================================

print("\n===================================")
print("DATASET CREATED SUCCESSFULLY")
print("===================================")

print(
    f"\nTotal Tulu Images : {total_tulu_images}"
)

print(
    f"Total Non-Tulu Images : {non_tulu_count}"
)

print("\nClasses:\n")

for cls in sorted(os.listdir(combined_path)):

    cls_path = os.path.join(
        combined_path,
        cls
    )

    print(
        f"{cls} -> {len(os.listdir(cls_path))} images"
    )

print(
    "\nTotal Classes:",
    len(os.listdir(combined_path))
)

# ============================================================
# STEP 7: IMAGE TRANSFORMS
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# STEP 8: LOAD DATASET
# ============================================================

dataset = ImageFolder(
    root=combined_path,
    transform=transform
)

print("\nDataset Loaded Successfully!")

print("\nClasses:\n")
print(dataset.classes)

num_classes = len(dataset.classes)

print("\nNumber of Classes:", num_classes)

# ============================================================
# STEP 9: SPLIT DATASET
# ============================================================

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("\nTraining Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))

# ============================================================
# STEP 10: DEVICE SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nUsing Device:", device)

# ============================================================
# STEP 11: LOAD CONVNEXT MODEL
# ============================================================

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=num_classes
)

model.to(device)

print("\nVit Model Loaded!")

# ============================================================
# STEP 12: LOSS FUNCTION & OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001
)

# ============================================================
# STEP 13: TRAIN MODEL WITH LIVE PROGRESS
# ============================================================

epochs = 10

for epoch in range(epochs):

    print(f"\n================ EPOCH {epoch+1}/{epochs} ================\n")

    # ================= TRAIN =================

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

        # LIVE BATCH PRINT
        if (batch_idx + 1) % 10 == 0:

            batch_accuracy = 100 * correct / total

            print(
                f"Batch [{batch_idx+1}/{len(train_loader)}] "
                f"| Loss: {loss.item():.4f} "
                f"| Accuracy: {batch_accuracy:.2f}%"
            )

    train_accuracy = 100 * correct / total

    # ================= VALIDATION =================

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_accuracy = 100 * val_correct / val_total

    print("\n========================================")

    print(f"Epoch [{epoch+1}/{epochs}] Completed")

    print(f"Train Loss       : {running_loss:.4f}")

    print(f"Train Accuracy   : {train_accuracy:.2f}%")

    print(f"Validation Acc   : {val_accuracy:.2f}%")

    print("========================================")

# ============================================================
# STEP 14: SAVE MODEL
# ============================================================

torch.save(model.state_dict(), "tulu_nontulu_vit.pth")

print("\nModel Saved Successfully!")

# ============================================================
# STEP 15: SAVE CLASS NAMES
# ============================================================

class_names = dataset.classes

with open("class_names.txt", "w") as f:

    for item in class_names:
        f.write(item + "\n")

print("Class Names Saved!")

Upload:
1. Tulu_Dataset_1.zip
2. Non Tulu.zip


Saving Non_Tulu.zip to Non_Tulu.zip
Saving Tulu_Dataset_1.zip to Tulu_Dataset_1.zip
Extracting Non_Tulu.zip...
Extracting Tulu_Dataset_1.zip...

Extraction Completed!

Folders inside dataset:

Tulu_Dataset_1
Non_Tulu

Tulu Root: /content/datasets/Tulu_Dataset_1/Tulu_Dataset_1

Processing Tulu Dataset...
ಜ -> 70 images
ಔ -> 73 images
ಮ -> 73 images
ಪ -> 72 images
ಕ -> 73 images
ಢ -> 67 images
ವ -> 74 images
ಙ -> 73 images
ಠ -> 77 images
ಧ -> 72 images
ಡ -> 73 images
ಥ -> 72 images
ಒ -> 73 images
ಸ -> 73 images
ಲ -> 73 images
ಳ -> 74 images
ಐ -> 73 images
ತ -> 72 images
ಝ -> 73 images
ಛ -> 73 images
ಷ -> 73 images
ಆ -> 74 images
ಗ -> 73 images
ಋ -> 73 images
ರ -> 73 images
ಅಃ -> 73 images
ಈ -> 73 images
ಊ -> 73 images
ಟ -> 75 images
ಓ -> 73 images
ೠ -> 67 images
ಬ -> 73 images
ಶ -> 64 images
ಞ -> 74 images
ಯ -> 70 images
ನ -> 72 images
ಅಂ -> 73 images
ಣ -> 73 images
ಎ -> 73 images
ಹ -> 73 images
ಇ -> 72 images
ಫ -> 72 images
ಏ -> 73 images
ಘ -> 70 images
ಖ -> 73 images
ಚ -> 73 images
ಉ -

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]


Vit Model Loaded!

================ EPOCH 1/10 ================

Batch [10/170] | Loss: 2.5797 | Accuracy: 40.94%
Batch [20/170] | Loss: 2.5418 | Accuracy: 45.16%
Batch [30/170] | Loss: 1.5610 | Accuracy: 47.29%
Batch [40/170] | Loss: 2.9901 | Accuracy: 46.95%
Batch [50/170] | Loss: 2.6482 | Accuracy: 47.12%
Batch [60/170] | Loss: 3.4032 | Accuracy: 47.14%
Batch [70/170] | Loss: 2.4773 | Accuracy: 46.88%
Batch [80/170] | Loss: 1.8286 | Accuracy: 46.52%
Batch [90/170] | Loss: 2.2146 | Accuracy: 46.42%
Batch [100/170] | Loss: 1.7122 | Accuracy: 46.50%
Batch [110/170] | Loss: 2.1369 | Accuracy: 46.73%
Batch [120/170] | Loss: 2.1529 | Accuracy: 46.95%
Batch [130/170] | Loss: 2.4410 | Accuracy: 46.90%
Batch [140/170] | Loss: 2.6152 | Accuracy: 46.79%
Batch [150/170] | Loss: 1.5268 | Accuracy: 47.06%
Batch [160/170] | Loss: 1.4314 | Accuracy: 48.03%
Batch [170/170] | Loss: 2.0382 | Accuracy: 48.45%

Epoch [1/10] Completed
Train Loss       : 392.2349
Train Accuracy   : 48.45%
Validation Acc 

In [ ]:
# ============================================================
# STEP 21: DOWNLOAD MODEL FILES
# ============================================================

files.download("tulu_nontulu_vit.pth")

files.download("class_names.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
from PIL import Image
import torch

# Upload image
uploaded = files.upload()

image_path = list(uploaded.keys())[0]

# Load image
image = Image.open(image_path).convert("RGB")

# Transform
image = transform(image)

# Add batch dimension
image = image.unsqueeze(0).to(device)

# Prediction
model.eval()

with torch.no_grad():

    outputs = model(image)

    probabilities = torch.softmax(outputs, dim=1)

    confidence, predicted = torch.max(probabilities, 1)

predicted_class = class_names[predicted.item()]
confidence_score = confidence.item() * 100

print("\n==============================")
print("RESULT")
print("==============================")

if predicted_class == "NON_TULU":

    print("Language   : NON_TULU")
    print(f"Confidence : {confidence_score:.2f}%")

else:

    print("Language   : TULU")
    print("Character  :", predicted_class)
    print(f"Confidence : {confidence_score:.2f}%")

Saving Screenshot 2026-06-02 100021.png to Screenshot 2026-06-02 100021.png

RESULT
Language   : TULU
Character  : ಡ
Confidence : 99.98%
